# Using Claude as a RAG Evaluation Judge

Evaluating RAG pipelines is one of the hardest problems in production AI. This cookbook shows how to use Claude to measure the quality of a RAG system across three key dimensions: **faithfulness**, **answer relevancy**, and **context precision**.

We'll build a mini evaluation harness from scratch using only the Anthropic Python SDK.

## Setup

Install the Anthropic SDK and set your API key.

In [ ]:
%pip install anthropic --quiet

In [ ]:
import os
import anthropic
import json

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-20250514"

## Sample RAG Pipeline

We'll define a simple dataset of questions, retrieved contexts, and generated answers to evaluate.

In [ ]:
eval_dataset = [
    {
        "question": "What is retrieval-augmented generation?",
        "context": "Retrieval-Augmented Generation (RAG) is an AI framework that combines a retrieval system with a generative language model. The retrieval component searches a knowledge base for relevant documents, which are then passed to the generative model as context to produce a grounded response.",
        "answer": "RAG is a technique that retrieves relevant documents from a knowledge base and uses them to help a language model generate more accurate, grounded responses.",
        "ground_truth": "RAG combines information retrieval with language generation to produce factually grounded responses."
    },
    {
        "question": "What are the main components of a transformer architecture?",
        "context": "The transformer architecture, introduced in 'Attention is All You Need', consists of an encoder and decoder. Key components include multi-head self-attention mechanisms, position-wise feed-forward networks, positional encodings, and layer normalization.",
        "answer": "Transformers use self-attention, feed-forward layers, and positional encodings.",
        "ground_truth": "Transformers consist of encoder/decoder blocks with multi-head attention, feed-forward networks, and positional encodings."
    },
    {
        "question": "How does vector search work?",
        "context": "Vector databases store embeddings — numerical representations of data. When a query arrives, it is converted to an embedding and compared against stored embeddings using similarity metrics like cosine similarity or dot product. The most similar vectors are returned as results.",
        "answer": "Vector search converts queries to embeddings and finds the closest stored embeddings using cosine similarity.",
        "ground_truth": "Vector search converts text to numerical embeddings and retrieves the most similar embeddings using distance metrics."
    }
]

print(f"Dataset loaded: {len(eval_dataset)} items")

## Metric 1: Faithfulness

Measures whether the generated answer is grounded in the retrieved context — no hallucinations.

In [ ]:
def evaluate_faithfulness(question: str, context: str, answer: str) -> float:
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question, retrieved context, and generated answer, evaluate whether the answer is faithful to the context (i.e., it only makes claims supported by the context).

Question: {question}
Context: {context}
Answer: {answer}

Faithfulness score criteria:
1.0 = Every claim in the answer is directly supported by the context
0.5 = Most claims are supported, but some minor unsupported statements
0.0 = The answer contains significant claims not found in the context

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    
    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Metric 2: Answer Relevancy

Measures whether the generated answer actually addresses the question asked.

In [ ]:
def evaluate_answer_relevancy(question: str, answer: str) -> float:
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question and an answer, evaluate how well the answer addresses the question.

Question: {question}
Answer: {answer}

Answer relevancy criteria:
1.0 = The answer directly and completely addresses the question
0.5 = The answer partially addresses the question or goes off-topic
0.0 = The answer does not address the question at all

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    
    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Metric 3: Context Precision

Measures whether the retrieved context is actually relevant to answering the question.

In [ ]:
def evaluate_context_precision(question: str, context: str) -> float:
    prompt = f"""You are an expert evaluator for RAG systems.

Given a question and retrieved context, evaluate how relevant and useful the context is for answering the question.

Question: {question}
Context: {context}

Context precision criteria:
1.0 = The context contains exactly what is needed to answer the question
0.5 = The context is somewhat relevant but contains a lot of noise
0.0 = The context is irrelevant to the question

Respond with ONLY a number between 0.0 and 1.0."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    
    try:
        return float(response.content[0].text.strip())
    except ValueError:
        return 0.0

## Run Evaluation

Now let's evaluate all items in the dataset across all three metrics.

In [ ]:
results = []

for item in eval_dataset:
    print(f"Evaluating: {item['question'][:50]}...")
    
    faithfulness = evaluate_faithfulness(item["question"], item["context"], item["answer"])
    relevancy = evaluate_answer_relevancy(item["question"], item["answer"])
    precision = evaluate_context_precision(item["question"], item["context"])
    
    results.append({
        "question": item["question"],
        "faithfulness": faithfulness,
        "answer_relevancy": relevancy,
        "context_precision": precision,
        "average": round((faithfulness + relevancy + precision) / 3, 3)
    })

print("
Evaluation complete.")

## Results

In [ ]:
print(f"
{'Question':<50} {'Faith':>6} {'Relev':>6} {'Prec':>6} {'Avg':>6}")
print("-" * 74)

total_faith, total_relev, total_prec = 0, 0, 0

for r in results:
    q = r["question"][:48] + ".." if len(r["question"]) > 48 else r["question"]
    print(f"{q:<50} {r['faithfulness']:>6.2f} {r['answer_relevancy']:>6.2f} {r['context_precision']:>6.2f} {r['average']:>6.2f}")
    total_faith += r["faithfulness"]
    total_relev += r["answer_relevancy"]
    total_prec += r["context_precision"]

n = len(results)
print("-" * 74)
print(f"{'AVERAGE':<50} {total_faith/n:>6.2f} {total_relev/n:>6.2f} {total_prec/n:>6.2f}")

## Best Practices

1. **Run on at least 20-50 samples** — small datasets produce unreliable averages.
2. **Use the same model for generation and evaluation carefully** — Claude evaluating Claude responses can introduce bias. Consider using a different model for evaluation than generation.
3. **Set temperature=0 for evaluation calls** — you want deterministic scores.
4. **Track scores over time** — the real value is spotting regressions when you update your pipeline.
5. **Combine with human review** — use these scores to flag low-quality samples for human review, not as a replacement.

## Next Steps

- Try the [rag-eval-toolkit](https://github.com/Abanoubr/rag-eval-toolkit) for a full-featured Python package wrapping these patterns
- Add `ContextRecallMetric` to compare against ground truth answers
- Extend with domain-specific metrics (e.g., medical accuracy, citation quality)